# 1. Why an AI engineer must understand distributed systems:

Many AI projects begin in a notebook:

```
Input
 ↓
Python function
 ↓
Model
 ↓
Output
```
This is useful for experimentation, but a production AI application rarely operates inside one process or one machine.

A real AI system may contain:

```
User
 ↓
Load Balancer
 ↓
API Service
 ↓
Authentication Service
 ↓
Agent Orchestrator
 ├── LLM Provider
 ├── Embedding Service
 ├── Vector Database
 ├── PostgreSQL
 ├── Redis
 ├── Tool APIs
 └── Message Queue
 ↓
Evaluation and Safety Services
 ↓
Logs, Metrics, and Traces
```
Even when your application source code looks simple, it depends on many remote systems.

For example, a RAG application may perform these operations for one user question:

```
1. Authenticate the user.
2. Load conversation state.
3. Rewrite the query.
4. Generate an embedding.
5. Search a vector database.
6. Search a keyword index.
7. Rerank results.
8. Call an LLM.
9. Validate the output.
10. Save teh response.
11. Record evaluation and tracing data.
12. Stream the result to the user.
```

Each operation may run:
    * On a differnt server;
    * In a different region;
    * Under a different organization;
    * With a different failure mode;
    * At a different speed;
    * With a different consistency model.
Therefore, an AI engieer who understands only models will struggle to build dependable AI products.
Distributed-systems knowledge helps you answer questions such as:

```
what happens when the LLM provider times out?

What happens when a worker processes the same job twice?

What happens when the database is updated but the vector index is not?

What happens when 10,000 users upload documents simultaneously?

What happens when one slow tool delays the entire agent?

What happens when retries mutliply traffic during an outage?

What happens when two agents modify the same workflow state?

What happens when a user disconnects during streaming?

What happens when a region becomes unavailable?

```
These are not unusual edge cases. They are normal conditions in production systems.



# 2. What is a distributed system?

A distributed system is a collection of independent computers that cooperate through a network to behave like one larger system.

A distributed system has several defining properties:

```
Multiple independent processes
+ 
Communication over a network.
+
Shared objectives
+
No perfectly shared global state
+
Independent failures
+
Concurrency
```
Examples include:

    * Cloud applications;
    * Microservice systems;
    * Databases with replicas;
    * Kubernetes clusters;
    * Distributed training systems;
    * Vector databases;
    * Message Queues;
    * Content-delivery networks.
    * Milti-region applications.
    * LLM inference clusters;
    * Distributed agent paltforms.

A key lesson is:


`A distributed system is not merely a large application. It is an application in which communication and failure occur across independently operating components.`

# 3. Distributed system versus parallel system:

These concepts overlap but are not identical.


* 3.1 Parallel computing:
Parallel computing focuses on performing multiple calculations at the same time, usually to complete one task faster.

Example:
```
Large matrix multiplication
 ↓
Split across 8 GPUs
 ↓
Combine results
```
The primary concern is computational speed.

# 3.2 Distributed computing 

Distribute computing focuses on coordinating independent machines through a network.

Example:

```
API server
 ↓
Remote inference server
 ↓
Remote database
 ↓
Remote queue
```
The primary concerns include:

    * Communication.
    * Partial Failure;
    * Consistency;
    * Coordination;
    * Availability;
    * Scalability;
A distributed trainging job is both parallel and distributed.
A web application using a remote is distributed even when it performs little parallel computation.

# 4. Distributed system versus microservices:

Microsservices are one possible way to organize a distributed system.
A distributed system may also be:

    * A replicated database;
    * A worker cluster;
    * A distributed cache;
    * A model-serving cluster;
    * A peer-to-peer system;
    * A message-processing pipeline.

A microservice architecture divides an application into independently deployed services.

Example:

```
User Service.
Document Service.
Embedding Service.
Retrieval Service.
Agent Service.
Billing Service.
```

Microservices add:
    * Network calls;
    * Deployment complexity;
    * Version compatibility problems;
    * Monitoring requirements;
    * Distributed Transactions;
    * Operational Overhead.

A modular monolith is often better for an early-stage AI product:

```
One deployable application
+ 
Clear internal modules
+ 
One primary database
+ 
Background workers where needed
```

The system can later be separated when scaling, ownership, or reliability requirements justify it.


# 5. The three fundamental difficulties:
Distributed systems are difficult because of three realities.


### 5.1 The network is unreliable
Messages may be:
    * Delayed;
    * Duplicated;
    * Dropped;
    * Recorded;
    * Corrupted;
    * Delivered after a timeout;
    * Delivered after the sender has already retired.

### 5.2 Components fail independently:

One service may fall while the rest continue operating.
This is called partial failure.

Example:

```
API service: healthy
PostgreSQL: healthy
Vector database: Unhealthy
LLM provider: slow
Redis: healthy
```
The overall system is neither fully working nor fulling broken.

### 5.3 There is no perfectly shared global clock:

Different machines have different clocks:

Clock differences may result from:
    * Hardware drift;
    * Delayed Synchronization;
    * Network latency;
    * Time corrections;
    * Virtual-machine behaviour;

Therefore, timestamps from differnt machines cannot always establish a perfectly reliable event order. These three realities lead to many of the core distributed-systems problems.


# 6. The fellacies of distributed computing:

Engineers often unconsciously make incorrect assumptions such as:

```
The network is reliable.

Latency is zero.

Bandwidth is infinite.

The network is secure.

The network topology never changes.

There is only one administrator.

Data transfer has no cost.

Every component is identical.
```
All of these assumptions eventually fail.

For AI systems, additional dangerous assumptions include:

```
The model provider will always respond.

Every model response will follow the schema.

The same model name always means identical behaviour.

A tool call that timed out did not execute.

The vector index is immediately synchronized with the database.

The user will keep the connection open.

A retired agent step is harmless.

A background worker processes each job exactly once.
```
A production design must explicitly reject theser assumptions.




# 7. A simple distributed AI request:

Consider a user asking:

    `Summarize the uploaded contract and identify risky clauses.`

A possible request path is:


```
Browser
   ↓
CDN or Reverse Proxy
   ↓
API Gateway
    ↓
Authentication
    ↓
Application Service
    ↓
Document Metadata Database
    ↓
Object Storage
    ↓
Parser Service
    ↓
Embedding Service
    ↓
Vector Database
    ↓
Reranker
    ↓
LLM Service
    ↓
Output Validator
    ↓
Audit Log
    ↓
Browser
```

Every arrow represents:
    * A network call;
    * A serialization boundary;
    * A timeout decision;
    * A possible retry;
    * A security boundary;
    * A latency contribution;
    * A possible failure.

This is the correct distributed-systems mental model:

Every remote call is slower; less reliable, and less predictable than a local function call.









# 8. Processes, nodes, and services:

A few important terms:

* Process:
A running instance of a program.

* Node:
A mchine or virtual machine participating in the system.

* Container:

An isolated application environment running on a node.

* Service:
A logical capability exposed through an interface.

* Instance:
One Running compy of service.

Example:

```
Retrieval Service
 ├── Instance 1
 ├── Instance 2
 ├── Instance 3
 └── Instance
```
A load balancer may distribute requests across these instances.

The distinction matters because:

```
Service = logical role.
Instance = One running copy.
Node= Machine hosting instances.
```

# 9. Communication models:
Distributed components communicate in several ways.

* 9.1 Synchronous request-response:

The caller waits for a result.

`API Service -> Retrieval Service -> Response`

Example:
    * HTTP REST;
    * gRPC;
    * Database queries;
    * Synchronous tool calls.

Advantages:
    * Simple mental model;
    * Immediate result;
    * Easier for short operations.
Disadvantages:
    * Caller is blocked;
    * Failure propagate;
    * Latency accumulates;
    * Slow downstream services slow the caller.

* 9.2 Asynchronous messaging:

The caller submits work and continues without waiting for completion.

`API Service -> Queue -> Worker`

Advantages:
    * Decoupling;
    * Better handling of spikes;
    * Retry support;
    * Long-running task support.
Disadvantages:
    * Eventual Consistency;
    * More operational complexity.
    * Hardar debugging;
    * Duplicate processing;
    * Job-status management.

* 9.3 Streaming:

Data is delivered incrementally over one connection or event stream.

Examples:
    * Tokem streaming;
    * Audio Streaming;
    * Event streams;
    * Model output streaming;
    * Change-data capture.

* 9.4 Publish-subscribe:

A publish emits events without addression one specific consumer.

```
        DocumentIndexed event
                 ↓
 ┌───────────────┬───────────────────┐
Analytics Cache Invalidator Audit Service
```
Publish-subscribe enables loosely coupled systems.


# 10. Remote procedure calls:

A remote procedure call, or RPC, makes a network operation resemble a local function call.

`result = embedding_service.embed(text)`

This looks like a normal function, but internally it may involve:

```
Serialize request
 ↓
Open or reuse connection
 ↓
Send bytes
 ↓
Route request
 ↓
Remote processing
 ↓
Serialize response
 ↓
Return across network
 ↓
Deserialize response
```

A remote call can fall in ways a local function cannot.

The caller may observe a timeout without knowing whether the remote operation:

    * Never started;
    * Started but failed;
    * Completed successfully;
    * Completed successfully but the response was lost.

This ambiguity is fundamental.


# 11. Serialization and contracts:

Distributed services exchange bytes, not Python objects.

Data must be serialized.

Common formats include:

    * JSON
    * Protocol Buffers;
    * MessagePack;
    * Avro;
    * Parquet for analytical data.

A service contract defines:
    * Request fields;
    * Reponse fields;
    * Data types;
    * Required and optional fields;
    * Error formats;
    * Versioning;
    * Maximum sizes.

Example:

```
{
"request_id": "req-812",
"model": "embedding-model-v2",
"texts": [
"First document chunk",
"Second document chunk"
]
}
```

Response:

```
{
"request_id": "req-812",
"vectors": [
[0.13, -0.82, 0.41],
[0.28, -0.14, 0.76]
],
"dimension": 3
}
```

Strong contracts reduce integration errors.
In Python, Pydantic can be used at service boundaries:

```Python
form pydantic import BaseModel, Field

class EmbeddingRequest(BaseModel):
    request_id : str
    texts: list[str] = Field(min_length=1, max_length=128)

class EmbeddingResponse(BaseModel):
    request_id: str
    vectors: list[list[float]]
    dimension: int
```
# 12. Backward-compatible API evolution:

Distributed services are often updated independently.

Suppose an old response is:

`
{
    "answer":"...."
}
`
This is usually backward-compatible because old clients can ignore the new field.

Removing or renaming `answer` may break existing clients.

Safer evolution patterns include:

    * Adding optional fields;
    * Providing defaults;
    * Supporting old and new versions temporarilyy;
    * Using explicit API versioning;
    * Avoiding changes in field meaning;
    * Testing contracts between producers and consumers.

Model and prompt outputs also need versioned contracts.

# 13. Latency:

Latency is the time required to complete and operation.

For a distributed request:

```
Total latency =
Client Network
+ Gateway
+ Authentication
+ Database
+ Retrieval
+ Reranking
+ Model
+ Tools 
+ Validation
+ Response transfer.

```
If stages are sequential, their latencies approximately add.

Example:

```
Gateway:                30 ms
Authentication:         40 ms
Database lookup:        50 ms
Embedding call:        150 ms
Vector search:          90 ms
Reranking:             180 ms
LLM first token:      1200 ms
Output validation:      60 ms
-----------------------------
Total first response: 1800 ms
```
The system should be optimized using measured latency, not intuition.

# 14. Percentiles and tail latency:

Average latency hides slow requests.

Suppose ten request latencies are:

`100, 110, 120, 125, 135, 140, 150, 180, 2000 ms`

The average is distorted by one slow request, while most requests are fast.

Production systems commonly measure:

```
p50: median request,
p90: 90% of requests,
p95: 95% are faster,
p99: 99% are faster
```

Tail latency refers to the slowest portion of requests.

It matters greatly in AI systems because one request may fan out to many services.

Suppose a system sends requests to 20 tools or shards. Even when each component is slow only occasionally, the complete request waits for the slowest component.

This is called tail-latency amplification.

# 15. Latency budgets:

A latency budget assigns a maximum time to each stage.

Suppose your user-facing goal is:
` First token within 2 seconds at p95`


You might allocate:

```
Gateway and authentication:         150 ms
Conversation-state loading:         100 ms
Retrieval and reranking:            400 ms
Prompt Construction:                 50 ms
Model queue and first token:       1200 ms
Safety checks:                      100 ms
------------------------------------------
Total Budget:                       2000 ms
```
A timeout should be derived from this budget.

A downstream call should not receive a 30-second when only 2 seconds remain in the user request.

# 16. Timeouts:

Every remote call requires a timeout.

Without a timeout, a request may wait indefinitely.

Example:

```Python

import httpx
async def call_reranker(payload: dict) -> dict:
    timeout = httpx.timeout(
        connect=2.0,
        read=5.0,
        write= 2.0,
        pool= 2.0,
    )
    async with httpx.AsyncClient(timeout= timeout) as client:
        response = await client.post(
            "https://reranker.internal/rerank",
            json=payload,
        )
        response.raise_for_status()
        return response.json()
```









# Different timeouts protect different stages:

    * connection timeout;
    * Read timeout;
    * Write timeout;
    * Connection-pool timeout;
    * Overall deadline;
Timeouts should be shorter for interactive operations and longer for background jobs.


# 17. Deadlines versus timeouts:

A timeout limits one operation.

A deadline limits the entire request.

Example:

`User request deadline: 5 seconds`

After authentication consumes 500 ms, the remaining budget is 4.5 seconds.

After retrieval consumes 1 second, the remaining budget is 3.5 seconds.

The remaining deadline should be propagated downstream.

This is better than assigning every service an independent 5-second timeout, which could create a much longer end-to-end request.


# 18. Retries:

Retries are useful for transient failures such as:
    * Temporary network errors;
    * Rate limits;
    * Overload services;
    * Short provider outages.

Retries are dangerous when:
    * The operation is not idempotent;
    * The failure is permanent;
    * Many callers retry simultaneously;
    * The downstream service is already overloaded;
    * The original operation may have succeeded.

A retry policy should define:

    ```
    Which errors are retryable?
    How many attempts are allowed?
    How long should the caller wait?
    Is the operation safe to repeat?
    What is the total retry budget?
    ```
# 19. Exponential backoff and jitter:

A naive retry may immediately repeat:

```
Attempt 1: now
Attempt 2: immediately
Attempt 3: immediately
```
This can overlaod a struggling service.
Exponential backoff increases the delay:

```
Attempt 1: 0 seconds
Attempt 2: 1 seconds
Attempt 3: 2 seconds
Attempt 4: 4 seconds
Attemtp 5: 8 seconds
```
Jitter adds randomness so that many clients do not retry at exactly the same time.

```Python
import asyncio
import random
from collections.abc import Awaitable, Callable
from typing import TypeVar

T = TypeVar("T")

async def retry_with_backoff(
    operation: Callable[[],Awaitable[T]],
    *,
    max_attempts: int =4,
    base_dealy: float=0.5,
    max_delay: flaot =0.8,
) -> T:
    last_error: Exception | None = None
    for attempt in range(max_attempts):
        try:
            return await operation()
        except (TimeoutError, ConnectionError) as error:
            last_error = error
            if attempt == max_attempts -1:
                break
            exponential_delay = min(
                base_delay * (2**attempt),
                max_delay,
            )
            jittered_delay = random.uniform(0, exponential_delay,)

            await asyncio.sleep(jittered_dealy)
    assert last_error is not None
    raise last_error
```

Do not retry every exception. Programming errors and invalid input usually require correction, not repetition.



# 20. Retry amplification:

Retries can multiply traffic:
Suppose:

```
API Gateway retries 3 times.
Application Service retries 3 times
Agent Service retries 3 times.
Tool Client retries 3 times
```

In the worst case, one request can produce:

`3x3x3x3=81 attempts`

This is retry amplification.
A struggling downstream service may be destroyed by the recovery mechanism intended to help it.

Better practice:
    * Retry at one carefully selected layer;
    * Use a shared deadline;
    * Limit total attempts;
    * Apply circuit breaking;
    * Monitor retry volume.

# 21. Idempotecy:

An opeartion is idempotent when repeating it produces the same intended outcome.

Idempotent:
`Set user status to "verified"`

Reapeating it still leaves the status as `verified`.

Not naturally idempotent:

`Charge the customer $100`

Repeating it may charge the customer twice.

AI agents are especially likely to repeat actions because of;
    * retries.
    * resumed workflows;
    * model self-correction;
    * Duplicate messages;
    * Timeouts;
    * Human approval workflows;
Therefore, write operations should accept an idempotency key.

Example Request:

```Python
{
    "idempotency_key":"order-thread-18-action-4",
    "customer_id":"customer-91",
    "amount": 10
}
```
The server records the key.
If the same key appears again, it returns the earlier result rather than repeating the action.

# 22. Idempotent action pattern:

A simplified database table:

```
idempotency_key
request_hash
status
response
created_at
```
Execution:

```
1. Receieve idempotency key.
2. Check whether the key already exists.
3. If completed, return the saved response.
4. If in progress, reject or wait.
5. If absent, reserve the key.
6. Execute the operation.
7. Save the final response.
8. Return the result.
```
This pattern is critical for:

* Payment tools.
* Email sending.
* Ticket creation.
* Document deletion.
* Claender booking.
* Database updates.
* Robotics actions.








# Circuit Breakers:

A circuit breaker prevents repeated calls to a failing service.
It has three conceptual states.

## Close:
Requests flow normally.

## Open:
Requests fail immediately without calling the unhealthy dependency.

## Half-Open:
A limited number of test requests are allowed to determine whether the service has recovered.

Flow:
```
Healthy⬇️